In [ ]:
# Cell 1: Install Required Packages
!pip install ASE
!pip install mace-torch ase rdkit weas-widget

In [14]:
# Cell 2: Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
from ase import Atoms
from ase.build import bulk, molecule
from mace.calculators import mace_mp, mace_off

print("All imports successful.")

All imports successful.


In [26]:
# Cell 3: Load MACE-OFF

print("Loading MACE-OFF (medium model)...")
calc_mol = mace_off(model="medium", default_dtype="float64")
print("MACE-OFF loaded.")

Loading MACE-OFF (medium model)...
Using MACE-OFF23 MODEL for MACECalculator with /Users/gregorysantilli/.cache/mace/MACE-OFF23_medium.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.
MACE-OFF loaded.


/Users/gregorysantilli/miniforge3/envs/molsim/lib/python3.11/site-packages/mace/calculators/mace.py:226: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(f=model_pat

In [32]:
# Cell 4: Optimize Nitrous Oxide (Gas)
from ase.optimize import BFGS

N_2_O = Atoms('NNO', positions=[(0,0,0), (1.13, 0, 0), (2.32, 0, 0)],
            cell=[15, 15, 15], pbc=False)
N_2_O.calc = calc_mol

opt = BFGS(N_2_O, logfile=None)
opt.run(fmax=0.001)

bond_length_NN = N_2_O.get_distance(0, 1)
bond_length_NO = N_2_O.get_distance(1, 2)

E_N_2_O = N_2_O.get_potential_energy()

print(f"Optimized NN bond length: {bond_length_NN:.4f} Å   (exp: 1.13 Å)")
print(f"Optimized NO bond length: {bond_length_NO:.4f} Å   (exp: 1.19 Å)")
print(f"N₂O total energy:          {E_N_2_O:.6f} eV")

Optimized NN bond length: 1.1179 Å   (exp: 1.13 Å)
Optimized NO bond length: 1.1869 Å   (exp: 1.19 Å)
N₂O total energy:          -5027.926705 eV


In [33]:
# Cell 5: Calculate Nitrous Oxide (gas) Properties
from ase.build import molecule
from ase.optimize import QuasiNewton
from ase.thermochemistry import IdealGasThermo
from ase.vibrations import Vibrations
from ase.units import kJ, mol

atoms_N2O = molecule('N2O')
atoms_N2O.calc = calc_mol
dyn = QuasiNewton(atoms_N2O, logfile=None)
dyn.run(fmax=0.01)
potentialenergy = atoms_N2O.get_potential_energy()

vib = Vibrations(atoms_N2O, name='n2o_vib')
vib.clean()
vib.run()
vib_energies = vib.get_energies()
vib_energies = np.array([e.real for e in vib_energies if e.real > 0.01])


thermo = IdealGasThermo(
    vib_energies=vib_energies,
    potentialenergy=potentialenergy,
    atoms=atoms_N2O,
    geometry='nonlinear', # Linear (Straight Line) or Nonlinear (Bent in any way)
    symmetrynumber=1, # How many times you can rotate the molecule and get the same configuration
    spin=0, # 0.5 for each unpaired electrons
)
H_N2O = thermo.get_enthalpy(temperature=298.15, verbose=False)

H_N2O_kJ = H_N2O * (1/(kJ/mol))

print(f"Enthalpy of N2O at 298 K: {H_N2O_kJ:.4f} kJ/mol")

Enthalpy of N2O at 298 K: -485086.0305 kJ/mol


In [34]:
# Cell 6: Calculate Enthalpy for N2

atoms_N2 = molecule('N2')
atoms_N2.calc = calc_mol
dyn = QuasiNewton(atoms_N2, logfile=None)
dyn.run(fmax=0.01)
potentialenergy = atoms_N2.get_potential_energy()

vib = Vibrations(atoms_N2, name='n2_vib')
vib.clean()
vib.run()
vib_energies = vib.get_energies()
vib_energies = np.array([e.real for e in vib_energies if e.real > 0.01])


thermo = IdealGasThermo(
    vib_energies=vib_energies,
    potentialenergy=potentialenergy,
    atoms=atoms_N2,
    geometry='linear', # Linear (Straight Line) or Nonlinear (Bent in any way)
    symmetrynumber=2, # How many times you can rotate the molecule and get the same configuration
    spin=0, # 0.5 for each unpaired electrons
)
#G_N2 = thermo.get_gibbs_energy(temperature=298.15, pressure=101325.0, verbose=False)
H_N2 = thermo.get_enthalpy(temperature=298.15, verbose=False)

H_N2_kJ = H_N2 * (1/(kJ/mol))

print(f"N2 Enthalpy at 298 K: {H_N2_kJ:.4f} kJ/mol")

N2 Enthalpy at 298 K: -287572.7094 kJ/mol


In [35]:
# Cell 7: Calculate Enthalpy for O2

atoms_O2 = molecule('O2')
atoms_O2.calc = calc_mol
dyn = QuasiNewton(atoms_O2, logfile=None)
dyn.run(fmax=0.01)
potentialenergy = atoms_O2.get_potential_energy()

vib = Vibrations(atoms_O2, name='o2_vib')
vib.clean()
vib.run()
vib_energies = vib.get_energies()
vib_energies = np.array([e.real for e in vib_energies if e.real > 0.01])


thermo = IdealGasThermo(
    vib_energies=vib_energies,
    potentialenergy=potentialenergy,
    atoms=atoms_O2,
    geometry='linear', # Linear (Straight Line) or Nonlinear (Bent in any way)
    symmetrynumber=2, # How many times you can rotate the molecule and get the same configuration
    spin=1, # 0.5 for each unpaired electrons
)
#G_O2 = thermo.get_gibbs_energy(temperature=298.15, pressure=101325.0, verbose=False)
H_O2 = thermo.get_enthalpy(temperature=298.15, verbose=False)

H_O2_kJ = H_O2 * (1/(kJ/mol))

print(f"O2 Enthalpy at 298 K: {H_O2_kJ:.4f} kJ/mol")

O2 Enthalpy at 298 K: -394851.2359 kJ/mol


In [36]:
# Cell 8: Calculate Enthalpy Change for N2O (g) using N2 and O2 and Error

dH_Exp = H_N2O_kJ - (H_N2_kJ + 0.5 * H_O2_kJ)
dH_Act = -82.05

print(f"Enthalpy change for N2O formation at 298 K: {(dH_Exp):.4f} kJ/mol")
print(f"Experimental: -82.05 kJ/mol")

Percent_Error = abs((dH_Exp - dH_Act) / dH_Act) * 100
print(f"Percent Error: {Percent_Error:.2f}%")

Enthalpy change for N2O formation at 298 K: -87.7031 kJ/mol
Experimental: -82.05 kJ/mol
Percent Error: 6.89%
